# Train PhoBertSpoTagger trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu/model từ Drive, cài thư viện) —
logic train thật sự nằm trong repo (`src/extraction/phobert_spo_tagger.py`,
`scripts/spo_extraction/train_phobert_spo_tagger.py`), đồng bộ qua git.

**Trước khi chạy:**
1. Đổi `DRIVE_ROOT` bên dưới nếu bạn để dữ liệu ở thư mục Drive khác.
2. Đã upload sẵn `data/dataset/simplificated_spo_sentence.csv` vào
   `DRIVE_ROOT/data/dataset/simplificated_spo_sentence.csv` trên Drive.
3. Chọn Runtime > Change runtime type > GPU trước khi chạy (nếu có GPU free trên Colab).


## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [2]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR


Cloning into '/content/CausalGraph'...
remote: Enumerating objects: 837, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 837 (delta 44), reused 67 (delta 22), pack-reused 718 (from 1)
Receiving objects: 100% (837/837), 1.84 MiB | 22.14 MiB/s, done.
Resolving deltas: 100% (427/427), done.
/content/CausalGraph


## 3. Gắn `data/` và `models/` vào Drive

Hai thư mục này bị `.gitignore`, không nằm trong git — clone xong sẽ trống hoặc không tồn tại.
Symlink sang Drive để dữ liệu và checkpoint được giữ lại qua các session, không cần copy tay
mỗi lần mở lại Colab.

In [3]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)

!rm -rf {REPO_DIR}/data {REPO_DIR}/models
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/models {REPO_DIR}/models

!ls -la {REPO_DIR}/data/dataset/ 2>/dev/null || echo "Chưa có data/dataset/simplificated_spo_sentence.csv trên Drive — upload trước khi train."


total 6419
-rw------- 1 root root 2926373 Aug  9 15:15 causal_sentences.csv
-rw------- 1 root root 3646112 Aug 10 01:34 simplificated_spo_sentence.csv


## 4. Cài thư viện

In [4]:
!pip install -q -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 136.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 26.6 MB/s eta 0:00:00
   ━

## 5. Đăng nhập Hugging Face Hub

Bắt buộc — cell train ở bước 6 giờ tự push model lên Hub ngay khi train xong
(dùng repo ID khai trong `configs/config.py`). Token tạo tại
https://huggingface.co/settings/tokens (quyền write).


In [5]:
from huggingface_hub import notebook_login

notebook_login()


## 6. Train

Checkpoint được lưu định kỳ vào `models/spo_tagger/` (= Drive, qua symlink ở bước 3).
Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần chạy lại cell này — `PhoBertSpoTagger.fit`
tự resume từ checkpoint gần nhất thay vì train lại từ đầu. Train xong sẽ tự
push model lên Hugging Face Hub (repo ID lấy từ configs/config.py).

In [6]:
!python -m scripts.spo_extraction.train_phobert_spo_tagger


config.json: 100% 557/557 [00:00<00:00, 2.78MB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 10.0MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 14.2MB/s]
tokenizer.json: 100% 3.13M/3.13M [00:00<00:00, 23.2MB/s]

pytorch_model.bin: downloading bytes:  67% 365M/543M [00:03<00:01, 99.1MB/s, 30.3MB/s  ]
pytorch_model.bin: reconstructing file:  62% 335M/543M [00:04<00:02, 82.7MB/s]
pytorch_model.bin: downloading bytes: 100% 366M/366M [00:04<00:00, 75.8MB/s, 30.4MB/s  ]
pytorch_model.bin: reconstructing file: 100% 543M/543M [00:04<00:00, 112MB/s, 47.0MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 35643.08it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTE